# Market Regime Detection and Strategy Case Study

## Questions

1. Detect market regimes from observable market features
2. Characterize those regimes and their return profiles
3. Investigate whether predictive signals vary across regimes
4. Build a regime-conditioned signal-based strategy
5. Conclude on whether the regime layer adds significant value

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from IPython.display import display
from scipy import stats
from sklearn.mixture import GaussianMixture

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Load data and perform a minimal audit

In [ ]:
df_raw = pd.read_csv("10_market_regime_detection_strategy_case.csv", parse_dates=["date"])
print("Raw shape:", df_raw.shape)
print("Duplicates:", int(df_raw.duplicated().sum()))
display(df_raw.isna().sum().to_frame("missing_count").T)
display(df_raw.head())

Daily data, 3,601 observations. Four features beyond date and index_close: **vix_like_index** (implied vol proxy), **credit_spread_change** (credit risk proxy), **market_breadth** (market participation), **term_spread_change** (rates/duration risk). One duplicate, 10 missing in market_breadth — minor.

In [ ]:
df = df_raw.sort_values("date").drop_duplicates().reset_index(drop=True).copy()

# Market breadth: 10 missing values. Forward-fill to avoid look-ahead bias
# (interpolate would use future values; ffill uses only past)
df["market_breadth"] = df["market_breadth"].ffill()

# Daily log return
df["ret_1d"] = np.log(df["index_close"]).diff()

# Target: next-day return (what we are trying to predict)
df["target"] = df["ret_1d"].shift(-1)

features = ["vix_like_index", "credit_spread_change", "market_breadth", "term_spread_change"]
model_df = df.dropna(subset=features + ["ret_1d", "target"]).copy()

print("Modeling rows:", len(model_df))
print("Date range:", model_df["date"].min().date(), "to", model_df["date"].max().date())

## 2. Reserve holdout before any analysis

In [ ]:
holdout = 600
dev_df = model_df.iloc[:-holdout].copy()
hold_df = model_df.iloc[-holdout:].copy()

print("Dev:", len(dev_df), "| Holdout:", len(hold_df))
print("Dev period:", dev_df["date"].min().date(), "to", dev_df["date"].max().date())
print("Holdout period:", hold_df["date"].min().date(), "to", hold_df["date"].max().date())

## 3. EDA on the development sample only

### 3.1 Time series overview

In [ ]:
%matplotlib inline

fig, axes = plt.subplots(3, 2, figsize=(13, 10))
axes[0,0].plot(dev_df["date"], dev_df["index_close"])
axes[0,0].set_title("Index level")
axes[0,1].plot(dev_df["date"], dev_df["ret_1d"])
axes[0,1].set_title("Daily return")

for ax, feat in zip(axes.ravel()[2:], features):
    ax.plot(dev_df["date"], dev_df[feat])
    ax.set_title(feat)

plt.tight_layout()
plt.show()

Clear alternation between calm and stress episodes. VIX spikes during stress, credit spread rises, breadth collapses, term spread compresses. The transitions are abrupt — the system switches between states rather than drifting gradually.

### 3.2 Feature space structure

Raw scatter plots of the four features — no coloring, no pre-classification.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

pairs = [
    ("credit_spread_change", "market_breadth"),
    ("credit_spread_change", "vix_like_index"),
    ("market_breadth", "vix_like_index"),
    ("market_breadth", "term_spread_change"),
    ("credit_spread_change", "term_spread_change"),
    ("vix_like_index", "term_spread_change"),
]

for ax, (xf, yf) in zip(axes.ravel(), pairs):
    ax.scatter(dev_df[xf], dev_df[yf], s=3, alpha=0.2, c="steelblue")
    ax.set_xlabel(xf)
    ax.set_ylabel(yf)
    ax.set_title(f"{xf} vs {yf}")

plt.tight_layout()
plt.show()

The scatter plots show a consistent pattern: a **dense core** (low credit spread, high breadth, low VIX) and a **diffuse tail** extending in the stress direction. This motivates at least 2 components in a mixture model. Whether 3 or more add value is an empirical question.

## 4. Choose the number of regimes

In [ ]:
X_dev = dev_df[features].copy()
X_dev_scaled = (X_dev - X_dev.mean()) / X_dev.std()

bic_scores = []
for k in range(2, 7):
    gmm_k = GaussianMixture(n_components=k, covariance_type="full", random_state=42, n_init=5)
    gmm_k.fit(X_dev_scaled)
    bic_scores.append({"k": k, "BIC": gmm_k.bic(X_dev_scaled), "AIC": gmm_k.aic(X_dev_scaled)})

bic_df = pd.DataFrame(bic_scores)
display(bic_df)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(bic_df["k"], bic_df["BIC"], marker="o", label="BIC")
ax.plot(bic_df["k"], bic_df["AIC"], marker="s", label="AIC")
ax.set_xlabel("Number of components")
ax.set_ylabel("Score (lower is better)")
ax.set_title("Model selection: BIC and AIC")
ax.legend()
plt.tight_layout()
plt.show()

# alternative simple: bic_df.plot(x="k", y=["AIC", "BIC"])

Both BIC and AIC are minimized at **k=3** (BIC = 27,382). Three regimes capture one more state than the visual two-regime reading: the third component likely represents a transition phase between calm and stress.

## 5. Explore and test the regimes

### 5b. Detecting and naming 

In [ ]:
best_k = 3
gmm = GaussianMixture(n_components=best_k, covariance_type="full", random_state=42, n_init=5)

X_dev = dev_df[features].copy()
X_dev_scaled = (X_dev - X_dev.mean()) / X_dev.std()
gmm.fit(X_dev_scaled)
dev_df["regime_raw"] = gmm.predict(X_dev_scaled)

# Name regimes by average VIX (ascending): calm, transition, stress
regime_vix = dev_df.groupby("regime_raw")["vix_like_index"].mean().sort_values()
name_map = {regime_vix.index[0]: "calm", regime_vix.index[1]: "transition", regime_vix.index[2]: "stress"}
dev_df["regime"] = dev_df["regime_raw"].map(name_map)

regime_stats = dev_df.groupby("regime").agg(
    ret_mean=("ret_1d", "mean"), ret_std=("ret_1d", "std"),
    vix_mean=("vix_like_index", "mean"), credit_mean=("credit_spread_change", "mean"),
    breadth_mean=("market_breadth", "mean"), term_mean=("term_spread_change", "mean"),
    n_days=("ret_1d", "count")
)
regime_stats["pct_of_total"] = (regime_stats["n_days"] / regime_stats["n_days"].sum() * 100).round(1)
display(regime_stats)

print("\nRegime mapping:", name_map)

In [ ]:
regime_col = dev_df["regime"].values
change_points = np.where(regime_col[1:] != regime_col[:-1])[0] + 1
change_points = np.concatenate([[0], change_points, [len(regime_col)]])

dur_records = []
for j in range(len(change_points) - 1):
    start = change_points[j]
    end = change_points[j + 1]
    dur_records.append({"regime": regime_col[start], "duration_days": end - start})

dur_df = pd.DataFrame(dur_records)
print("Average regime duration (days):")
display(dur_df.groupby("regime")["duration_days"].agg(["mean", "median", "max", "count"]).rename(columns={"count": "n_episodes"}))

Three regimes with distinct profiles:

- **Calm** (1,325 days, 44%): VIX ~14, breadth 0.65, vol 0.71%. Tight distribution, near-zero mean return (Sharpe +0.27). Average episode: 9.3 days, max 93 days. The normal state.
- **Transition** (901 days, 30%): VIX ~21, breadth 0.46, vol 1.15%. Slightly negative drift (Sharpe -0.55). Average episode: 4.6 days. Markets deteriorating but not yet in full stress.
- **Stress** (772 days, 26%): VIX ~31, breadth 0.28, credit spread 0.30, vol 1.81%. Significantly negative mean return (Sharpe -1.28). Average episode: 6.9 days (median 5, max 27).

All three are well-balanced (>25% each) and sufficiently persistent for a daily strategy.

### 5b. Regimes in feature space

In [ ]:
regime_colors = {"calm": "green", "transition": "orange", "stress": "red"}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
pairs = [
    ("credit_spread_change", "market_breadth"),
    ("credit_spread_change", "vix_like_index"),
    ("market_breadth", "vix_like_index"),
    ("market_breadth", "term_spread_change"),
    ("credit_spread_change", "term_spread_change"),
    ("vix_like_index", "term_spread_change"),
]
for ax, (xf, yf) in zip(axes.ravel(), pairs):
    for r in ["calm", "transition", "stress"]:
        mask = dev_df["regime"] == r
        ax.scatter(dev_df.loc[mask, xf], dev_df.loc[mask, yf],
                   s=4, alpha=0.3, c=regime_colors[r], label=r)
    ax.set_xlabel(xf); ax.set_ylabel(yf)
    ax.set_title(f"{xf} vs {yf}")
axes[0,0].legend(markerscale=3, fontsize=7)
plt.tight_layout()
plt.show()

The regime coloring confirms that the GMM has found a meaningful partition: **calm** (green) occupies the dense core, **stress** (red) covers the diffuse tail, and **transition** (orange) fills the overlap zone. This aligns with the raw EDA structure.

### 5c. Regime stability across dev and holdout

Before exploring return profiles and signals, I check whether the GMM regime boundaries generalise to the holdout. If the feature profiles within each regime shift materially, any downstream strategy built on regime-conditioned signals would be unreliable.

In [ ]:
# Apply regime detection to holdout using dev scaling parameters
X_hold_regime = (hold_df[features] - dev_df[features].mean()) / dev_df[features].std()
hold_df["regime_raw"] = gmm.predict(X_hold_regime)
hold_df["regime"] = hold_df["regime_raw"].map(name_map)

# Compare feature profiles
dev_chars = dev_df.groupby("regime")[features + ["ret_1d"]].mean()
hold_chars = hold_df.groupby("regime")[features + ["ret_1d"]].mean()
comparison = dev_chars.rename(columns=lambda c: c + "_dev").join(
    hold_chars.rename(columns=lambda c: c + "_hold"), how="outer"
)
display(comparison)

print("\nHoldout regime distribution:")
display(hold_df.groupby("regime").size().to_frame("n_days"))

The feature profiles are remarkably stable: calm retains VIX ≈ 14 and breadth ≈ 0.65 on both dev and holdout; stress retains VIX ≈ 31 and breadth ≈ 0.28; transition sits at VIX ≈ 21 and breadth ≈ 0.46 on both. The GMM boundaries generalise cleanly.

The return profiles, however, shift: all three regimes have positive mean returns on the holdout (calm +0.09%, stress +0.06%, transition +0.04%) while stress was significantly negative on the dev (−0.15%). This confirms that regime mean returns are not structural constants — they depend on the macro environment. A strategy that simply bets on the regime's historical direction would break down. The correct use of regimes is to condition *signals*, not to trade the regime label itself.

## 6. Return distributions by regime

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, regime in zip(axes, ["calm", "transition", "stress"]):
    rets = dev_df.loc[dev_df["regime"] == regime, "ret_1d"]
    ax.hist(rets, bins=50, edgecolor="white", alpha=0.7, color=regime_colors[regime])
    ax.axvline(rets.mean(), color="black", linestyle="--", label=f"mean = {rets.mean()*100:.2f}%")
    ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(f"{regime} (n={len(rets)}, vol={rets.std()*100:.2f}%)")
    ax.set_xlabel("Daily return")
    ax.legend(fontsize=8)
plt.suptitle("Return distribution by regime", y=1.02)
plt.tight_layout()
plt.show()

print("Mean return significance by regime:")
for regime in ["calm", "transition", "stress"]:
    rets = dev_df.loc[dev_df["regime"] == regime, "ret_1d"]
    t, p = stats.ttest_1samp(rets, 0)
    print(f"  {regime}: mean = {rets.mean()*100:.3f}%, vol = {rets.std()*100:.2f}%, t = {t:.2f}, p = {p:.4f}")

The return distributions on the development sample show:

- **Calm**: mean +0.012%, vol 0.71%, t = 0.61, p = 0.54. Not significantly different from zero.
- **Transition**: mean -0.040%, vol 1.15%, t = -1.04, p = 0.30. Not significant.
- **Stress**: mean -0.146%, vol 1.81%, t = -2.24, p = 0.025. Significantly negative — but the daily vol is 12x the daily mean. The signal-to-noise ratio is very poor.

This rules out a naive regime → direction strategy. Even where the mean is significant (stress), the distribution is wide enough that individual-day returns are essentially unpredictable from the regime alone. The regime identifies *what kind of market we are in*, not *which direction it will go tomorrow*.

A more robust use of the regime layer: check whether **predictive signals** behave differently across regimes. If certain lagged variables predict tomorrow's return more reliably in one regime, the regime tells us *which model to trust*.

## 7. Signal analysis: overall and by regime

I construct lagged candidate signals and check their predictive power — first overall, then by regime. The question is: does regime conditioning sharpen any signal materially?

In [ ]:
# Construct lagged signals — all use only past data
dev_df["lag_ret"] = dev_df["ret_1d"].shift(1)
dev_df["mom_5d"] = dev_df["ret_1d"].rolling(5).mean().shift(1)
dev_df["mom_20d"] = dev_df["ret_1d"].rolling(20).mean().shift(1)
dev_df["vol_ratio"] = (dev_df["ret_1d"].rolling(5).std() / dev_df["ret_1d"].rolling(20).std()).shift(1)
dev_df["breadth_lag"] = dev_df["market_breadth"].shift(1)
dev_df["vix_lag"] = dev_df["vix_like_index"].shift(1)

signals = ["lag_ret", "mom_5d", "mom_20d", "vol_ratio", "breadth_lag", "vix_lag"]
dev_signals = dev_df.dropna(subset=signals + ["target", "regime"]).copy()

# STEP 1: Overall correlations (no regime conditioning)
print("=== Overall signal correlations with next-day return ===")
overall_corrs = {}
for sig in signals:
    corr = dev_signals[sig].corr(dev_signals["target"])
    overall_corrs[sig] = round(corr, 4)
    print(f"  {sig}: {corr:.4f}")

# STEP 2: By regime
print("\n=== Signal correlations by regime ===")
corr_rows = []
for sig in signals:
    row = {"signal": sig, "overall": overall_corrs[sig]}
    for regime in ["calm", "transition", "stress"]:
        sub = dev_signals[dev_signals["regime"] == regime]
        row[regime] = round(sub[sig].corr(sub["target"]), 4)
    corr_rows.append(row)

corr_df = pd.DataFrame(corr_rows).set_index("signal")
display(corr_df)

# Highlight: where does regime conditioning help?
print("\n=== Regime conditioning value ===")
print("(comparing max |regime corr| vs |overall corr| for each signal)")
for sig in signals:
    overall_abs = abs(corr_df.loc[sig, "overall"])
    max_regime_abs = corr_df.loc[sig, ["calm", "transition", "stress"]].abs().max()
    improvement = max_regime_abs - overall_abs
    best_regime = corr_df.loc[sig, ["calm", "transition", "stress"]].abs().idxmax()
    print(f"  {sig}: overall |r| = {overall_abs:.4f}, best regime = {best_regime} |r| = {max_regime_abs:.4f}, improvement = {improvement:+.4f}")

**Overall signals (unconditional):** vix_lag is the strongest predictor (-0.054), followed by breadth_lag (+0.030), mom_20d (-0.018) and mom_5d (-0.018). These are weak but non-trivial — typical of daily equity return prediction.

**Regime-conditional signals — the key findings:**

- **mom_5d** triples in strength in calm (-0.052) and transition (-0.050) vs overall (-0.018), but vanishes in stress (+0.007). Short-term momentum works when markets are orderly but breaks down during stress.
- **mom_20d** doubles in calm (-0.043 vs -0.018 overall), fades in transition (-0.025) and further in stress (-0.013). Longer-term momentum is a calm-regime phenomenon.
- **vix_lag** flips sign: +0.013 in calm (high VIX yesterday → bounce today) vs -0.051 in transition and -0.058 in stress (high VIX → continued weakness). This is the most regime-dependent signal.
- **breadth_lag** also flips: -0.025 in calm vs +0.032 in transition. In calm, high breadth yesterday predicts a slight mean-reversion; in transition, it predicts continuation.
- **vol_ratio** and **lag_ret** are weak everywhere.

**Several signals have comparable strength within each regime** (e.g. in calm: mom_5d at -0.052, mom_20d at -0.043, breadth_lag at -0.025). Picking a single best signal discards useful information. A composite model (Ridge regression on all signals) per regime should outperform single-signal selection by combining these correlated but not identical predictors.

## 8. Backtest helper

A single helper function that computes all performance metrics including transaction costs. The cost model is simple: each day, the strategy pays `cost_per_unit × |Δexposure|`. This is deducted from the strategy return before computing net metrics.

No separate "turnover section" — costs are baked into the performance table from the start.

In [ ]:
# ---------------------------------------------------------------------------
# Backtest helper — gross and net performance in one call
# ---------------------------------------------------------------------------

def backtest(returns, exposure, name="strategy", cost_per_unit=0.0005):
    """Annualized performance metrics, gross and net of transaction costs.

    Parameters
    ----------
    returns     : pd.Series — daily strategy returns (exposure × target)
    exposure    : pd.Series — daily position, aligned to returns
    cost_per_unit : float   — proportional cost per unit of |Δexposure|
                              (0.0005 = 5 bps, reasonable for equity-index futures)

    Returns
    -------
    dict with gross metrics, turnover, costs, and net Sharpe.
    """
    # Gross
    mean_g   = returns.mean()
    std_g    = returns.std()
    cum_g    = (1 + returns).cumprod()
    dd_g     = (cum_g / cum_g.cummax() - 1).min()
    sharpe_g = np.sqrt(252) * mean_g / std_g if std_g > 0 else 0.0

    # Turnover & costs
    turnover_daily = exposure.diff().abs()
    cost_daily     = turnover_daily * cost_per_unit
    ann_turnover   = turnover_daily.mean() * 252

    # Net
    net_ret  = returns - cost_daily
    mean_n   = net_ret.mean()
    std_n    = net_ret.std()
    sharpe_n = np.sqrt(252) * mean_n / std_n if std_n > 0 else 0.0

    return {
        "strategy":       name,
        "ann_return":     (1 + mean_g)**252 - 1,
        "ann_vol":        std_g * np.sqrt(252),
        "sharpe_gross":   round(sharpe_g, 3),
        "sharpe_net":     round(sharpe_n, 3),
        "sharpe_haircut": round(sharpe_g - sharpe_n, 3),
        "max_drawdown":   dd_g,
        "ann_turnover":   round(ann_turnover, 1),
        "ann_cost_bps":   round(cost_daily.mean() * 252 * 10000, 1),
        "n_days":         len(returns),
    }


def backtest_passive(returns, name="buy_hold"):
    """Convenience wrapper for passive strategies (exposure = 1, no costs)."""
    mean_d = returns.mean()
    std_d  = returns.std()
    cum    = (1 + returns).cumprod()
    dd     = (cum / cum.cummax() - 1).min()
    sharpe = np.sqrt(252) * mean_d / std_d if std_d > 0 else 0.0
    return {
        "strategy":       name,
        "ann_return":     (1 + mean_d)**252 - 1,
        "ann_vol":        std_d * np.sqrt(252),
        "sharpe_gross":   round(sharpe, 3),
        "sharpe_net":     round(sharpe, 3),
        "sharpe_haircut": 0.0,
        "max_drawdown":   dd,
        "ann_turnover":   0.0,
        "ann_cost_bps":   0.0,
        "n_days":         len(returns),
    }


print("Backtest helpers defined: backtest(), backtest_passive()")

## 9. Regime-conditioned models: Ridge vs Random Forest (GridSearchCV)

For each regime I fit two models on all 6 signals:

- **Ridge** (linear, regularised): optimal linear combination, interpretable coefficients. Alpha selected by GridSearchCV with TimeSeriesSplit.
- **Random Forest** (non-linear, ensemble): can capture interactions. Hyperparameters selected by GridSearchCV with TimeSeriesSplit.

The RF grid is deliberately constrained — shallow trees (max_depth ≤ 3) and large leaf sizes (min_samples_leaf ≥ 50) — to limit overfitting on small per-regime samples (772–1305 observations).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# ---------------------------------------------------------------------------
# Parameter grids — intentionally tight to avoid overfitting
# ---------------------------------------------------------------------------
ridge_param_grid = {"alpha": [10, 100, 1000, 5000, 10000]}

rf_param_grid = {
    "max_depth":        [2, 3],
    "min_samples_leaf": [50, 80, 120],
    "max_features":     [2, 3],
}

tscv = TimeSeriesSplit(n_splits=5)

ridge_models = {}
rf_models    = {}
scalers      = {}

for regime in ["calm", "transition", "stress"]:
    mask = dev_signals["regime"] == regime
    X = dev_signals.loc[mask, signals].values
    y = dev_signals.loc[mask, "target"].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    scalers[regime] = scaler

    # --- Ridge GridSearch ---
    gs_ridge = GridSearchCV(Ridge(), ridge_param_grid, cv=tscv,
                            scoring="neg_mean_squared_error", n_jobs=-1)
    gs_ridge.fit(X_scaled, y)
    ridge_models[regime] = gs_ridge.best_estimator_
    pred_ridge = gs_ridge.predict(X_scaled)
    corr_ridge = np.corrcoef(pred_ridge, y)[0, 1]

    # --- RF GridSearch ---
    gs_rf = GridSearchCV(
        RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
        rf_param_grid, cv=tscv, scoring="neg_mean_squared_error", n_jobs=1
    )
    gs_rf.fit(X_scaled, y)
    rf_models[regime] = gs_rf.best_estimator_
    pred_rf = gs_rf.predict(X_scaled)
    corr_rf = np.corrcoef(pred_rf, y)[0, 1]

    print(f"--- {regime} (n={mask.sum()}) ---")
    print(f"  Ridge : best alpha={gs_ridge.best_params_['alpha']}, "
          f"CV MSE={gs_ridge.best_score_:.8f}, in-sample r={corr_ridge:.4f}")
    coef_s = pd.Series(gs_ridge.best_estimator_.coef_, index=signals)
    top3 = coef_s.abs().nlargest(3).index
    print(f"    Top coefs: { {s: round(coef_s[s], 6) for s in top3} }")

    print(f"  RF    : best params={gs_rf.best_params_}, "
          f"CV MSE={gs_rf.best_score_:.8f}, in-sample r={corr_rf:.4f}")
    imp_s = pd.Series(gs_rf.best_estimator_.feature_importances_, index=signals)
    top3_rf = imp_s.nlargest(3).index
    print(f"    Top importances: { {s: round(imp_s[s], 4) for s in top3_rf} }")
    print()

### 9b. Unconditional benchmarks: Ridge and RF without regime conditioning

To isolate the value added by the regime layer, I fit the same Ridge and RF (same GridSearch, same signals) on the entire dev set — no regime split. If the regime-conditioned models outperform these benchmarks on the holdout, the regime layer is adding genuine value.

In [ ]:
# ---------------------------------------------------------------------------
# Unconditional (pooled) Ridge and RF — same GridSearch setup
# ---------------------------------------------------------------------------
X_all = dev_signals[signals].values
y_all = dev_signals["target"].values

scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

gs_ridge_unc = GridSearchCV(Ridge(), ridge_param_grid, cv=tscv,
                            scoring="neg_mean_squared_error", n_jobs=-1)
gs_ridge_unc.fit(X_all_scaled, y_all)
ridge_unc = gs_ridge_unc.best_estimator_

gs_rf_unc = GridSearchCV(
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    rf_param_grid, cv=tscv, scoring="neg_mean_squared_error", n_jobs=1
)
gs_rf_unc.fit(X_all_scaled, y_all)
rf_unc = gs_rf_unc.best_estimator_

pred_ridge_unc = gs_ridge_unc.predict(X_all_scaled)
pred_rf_unc    = gs_rf_unc.predict(X_all_scaled)

print("Unconditional Ridge:")
print(f"  best alpha={gs_ridge_unc.best_params_['alpha']}, "
      f"in-sample r={np.corrcoef(pred_ridge_unc, y_all)[0,1]:.4f}")
coef_unc = pd.Series(ridge_unc.coef_, index=signals)
top3_unc = coef_unc.abs().nlargest(3).index
print(f"  Top coefs: { {s: round(coef_unc[s], 6) for s in top3_unc} }")

print(f"\nUnconditional RF:")
print(f"  best params={gs_rf_unc.best_params_}, "
      f"in-sample r={np.corrcoef(pred_rf_unc, y_all)[0,1]:.4f}")
imp_unc = pd.Series(rf_unc.feature_importances_, index=signals)
top3_rf_unc = imp_unc.nlargest(3).index
print(f"  Top importances: { {s: round(imp_unc[s], 4) for s in top3_rf_unc} }")

## 10. Development performance — all strategies

Five strategies compared, all with transaction costs baked in:

1. **Buy & hold** — passive benchmark
2. **Ridge uncond.** — same signals, no regime split
3. **RF uncond.** — same signals, no regime split
4. **Regime-Ridge** — one Ridge per regime
5. **Regime-RF** — one RF per regime

In [ ]:
# ---------------------------------------------------------------------------
# Apply models and build exposure / strategy returns
# ---------------------------------------------------------------------------

def apply_regime_models(strat_df, models, scalers, signals, dev_strat_ref=None):
    """Apply regime-specific models, z-score predictions within each regime."""
    exposure = pd.Series(0.0, index=strat_df.index)
    for regime in ["calm", "transition", "stress"]:
        mask = strat_df["regime"] == regime
        if mask.sum() == 0:
            continue
        X_sc = scalers[regime].transform(strat_df.loc[mask, signals].values)
        pred = models[regime].predict(X_sc)
        if dev_strat_ref is not None:
            dev_mask = dev_strat_ref["regime"] == regime
            dev_X = scalers[regime].transform(dev_strat_ref.loc[dev_mask, signals].values)
            dev_pred = models[regime].predict(dev_X)
            mu, sigma = dev_pred.mean(), dev_pred.std()
        else:
            mu, sigma = pred.mean(), pred.std()
        pred_z = (pred - mu) / sigma if sigma > 0 else pred * 0
        exposure.loc[mask] = np.clip(pred_z, -1, 1)
    return exposure


def apply_unconditional_model(strat_df, model, scaler, signals, dev_strat_ref=None):
    """Apply a single (unconditional) model, z-score predictions globally."""
    X_sc = scaler.transform(strat_df[signals].values)
    pred = model.predict(X_sc)
    if dev_strat_ref is not None:
        dev_X = scaler.transform(dev_strat_ref[signals].values)
        dev_pred = model.predict(dev_X)
        mu, sigma = dev_pred.mean(), dev_pred.std()
    else:
        mu, sigma = pred.mean(), pred.std()
    pred_z = (pred - mu) / sigma if sigma > 0 else pred * 0
    return pd.Series(np.clip(pred_z, -1, 1), index=strat_df.index)


# --- Dev exposures ---
dev_strat = dev_signals.copy()
dev_strat["exp_ridge"]     = apply_regime_models(dev_strat, ridge_models, scalers, signals)
dev_strat["exp_rf"]        = apply_regime_models(dev_strat, rf_models, scalers, signals)
dev_strat["exp_ridge_unc"] = apply_unconditional_model(dev_strat, ridge_unc, scaler_all, signals)
dev_strat["exp_rf_unc"]    = apply_unconditional_model(dev_strat, rf_unc, scaler_all, signals)

for tag, exp_col in [("ridge", "exp_ridge"), ("rf", "exp_rf"),
                      ("ridge_unc", "exp_ridge_unc"), ("rf_unc", "exp_rf_unc")]:
    dev_strat[f"ret_{tag}"] = dev_strat[exp_col] * dev_strat["target"]

# --- Dev performance table (costs included) ---
dev_perf = pd.DataFrame([
    backtest_passive(dev_strat["target"], "buy_hold"),
    backtest(dev_strat["ret_ridge_unc"], dev_strat["exp_ridge_unc"], "ridge_uncond"),
    backtest(dev_strat["ret_rf_unc"],    dev_strat["exp_rf_unc"],    "rf_uncond"),
    backtest(dev_strat["ret_ridge"],     dev_strat["exp_ridge"],     "regime_ridge"),
    backtest(dev_strat["ret_rf"],        dev_strat["exp_rf"],        "regime_rf"),
])
display(dev_perf)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
plot_specs = [
    ("target",       "Buy & Hold",       "gray",       "-"),
    ("ret_ridge_unc","Ridge uncond.",     "steelblue",  "--"),
    ("ret_rf_unc",   "RF uncond.",        "darkorange", "--"),
    ("ret_ridge",    "Regime-Ridge",      "steelblue",  "-"),
    ("ret_rf",       "Regime-RF",         "darkorange", "-"),
]
for col, label, color, ls in plot_specs:
    cum = (1 + dev_strat[col]).cumprod()
    ax.plot(dev_strat["date"].values, cum.values, label=label, color=color, ls=ls, lw=1.2)
ax.legend(fontsize=8)
ax.set_title("Development: cumulative returns — all strategies (gross)")
plt.tight_layout()
plt.show()

### Dev results

On the development set (2010–2021), all four signal-based strategies beat buy-and-hold (Sharpe −0.59, dragged down by an −84% drawdown).

**Regime-RF shows an in-sample Sharpe of ~3.5 — this is almost certainly overfitting.** Even with constrained depth (max_depth ≤ 3) and large leaves (min_samples_leaf ≥ 50), the RF achieves in-sample correlations of 0.22–0.28 per regime versus 0.06 for the Ridge. With daily signal correlations of ~5%, an in-sample r of 0.25 is implausibly high: the model is fitting noise patterns within each regime. This will show up as a large dev-to-holdout Sharpe drop.

The regime-Ridge (Sharpe ~1.0 gross, ~0.5 net) is more realistic but still optimistic in-sample. The unconditional Ridge (Sharpe ~0.7 gross, ~0.4 net) is the most conservative benchmark. The gap between regime-Ridge and unconditional Ridge reflects the regime layer's *in-sample* contribution — the holdout will tell whether it holds.

Turnover is materially higher for regime-conditioned models (~135–158 units/year) than unconditional ones (~86–89), because regime switches force abrupt repositioning. This translates to a Sharpe haircut of ~0.5 for the regime models vs ~0.3 for unconditioned — regime conditioning is more expensive to implement.

## 11. Holdout evaluation

In [ ]:
# --- Construct signals on holdout ---
hold_df["lag_ret"]     = hold_df["ret_1d"].shift(1)
hold_df["mom_5d"]      = hold_df["ret_1d"].rolling(5).mean().shift(1)
hold_df["mom_20d"]     = hold_df["ret_1d"].rolling(20).mean().shift(1)
hold_df["vol_ratio"]   = (hold_df["ret_1d"].rolling(5).std() /
                          hold_df["ret_1d"].rolling(20).std()).shift(1)
hold_df["breadth_lag"] = hold_df["market_breadth"].shift(1)
hold_df["vix_lag"]     = hold_df["vix_like_index"].shift(1)

hold_strat = hold_df.dropna(subset=signals + ["target", "regime"]).copy()

# --- Apply all four models from dev ---
hold_strat["exp_ridge"]     = apply_regime_models(hold_strat, ridge_models, scalers, signals, dev_strat_ref=dev_strat)
hold_strat["exp_rf"]        = apply_regime_models(hold_strat, rf_models, scalers, signals, dev_strat_ref=dev_strat)
hold_strat["exp_ridge_unc"] = apply_unconditional_model(hold_strat, ridge_unc, scaler_all, signals, dev_strat_ref=dev_strat)
hold_strat["exp_rf_unc"]    = apply_unconditional_model(hold_strat, rf_unc, scaler_all, signals, dev_strat_ref=dev_strat)

for tag, exp_col in [("ridge", "exp_ridge"), ("rf", "exp_rf"),
                      ("ridge_unc", "exp_ridge_unc"), ("rf_unc", "exp_rf_unc")]:
    hold_strat[f"ret_{tag}"] = hold_strat[exp_col] * hold_strat["target"]

# --- Holdout performance (costs included) ---
hold_perf = pd.DataFrame([
    backtest_passive(hold_strat["target"], "buy_hold"),
    backtest(hold_strat["ret_ridge_unc"], hold_strat["exp_ridge_unc"], "ridge_uncond"),
    backtest(hold_strat["ret_rf_unc"],    hold_strat["exp_rf_unc"],    "rf_uncond"),
    backtest(hold_strat["ret_ridge"],     hold_strat["exp_ridge"],     "regime_ridge"),
    backtest(hold_strat["ret_rf"],        hold_strat["exp_rf"],        "regime_rf"),
])
display(hold_perf)

print("\nHoldout regime distribution:")
display(hold_strat["regime"].value_counts().to_frame("n_days"))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
for col, label, color, ls in plot_specs:
    cum = (1 + hold_strat[col]).cumprod()
    ax.plot(hold_strat["date"].values, cum.values, label=label, color=color, ls=ls, lw=1.2)
ax.legend(fontsize=8)
ax.set_title("Holdout: cumulative returns — all strategies (gross)")
plt.tight_layout()
plt.show()

### Holdout results

The holdout (2021–2023, ~580 days) confirms the overfitting diagnosis:

**Regime-RF:** Sharpe drops from ~3.5 (dev) to ~1.6 (holdout gross) — a 55% collapse. Even so, the holdout gross Sharpe of 1.6 may look respectable. But this RF was trained on per-regime subsets of 772–1305 obs with in-sample r of 0.22–0.28 where the true signal is ~5% — the model is memorising tree splits that happen to work out-of-sample only partially. Net of costs (turnover ~175, haircut ~0.6), the net Sharpe drops to ~1.0. Still positive, but far less impressive than the in-sample 3.5 suggested.

**Regime-Ridge:** Sharpe drops from ~1.0 to ~0.6 (holdout gross). After the cost haircut (~0.5), the net Sharpe is close to zero. The regime-Ridge barely breaks even after costs on the holdout.

**Unconditional models:** The unconditional RF (Sharpe ~1.2 gross, ~0.9 net) actually outperforms the regime-RF net on holdout when accounting for the regime models' higher turnover. The unconditional Ridge (Sharpe ~0.65 gross, ~0.35 net) also outperforms the regime-Ridge net.

**The regime layer does not add value after costs.** The extra turnover from regime switching (~60-70 more annual units than unconditional) eats into whatever marginal signal improvement the regime split provides. The unconditional models, being smoother and cheaper to trade, deliver better net performance.

**Buy-and-hold** (Sharpe 0.87) outperforms all strategies except the RF variants. In a positive equity market, the cost of being partially hedged exceeds the signal's alpha.

## 12. Sharpe stability by regime

For each regime, I compute the strategy's Sharpe restricted to days classified in that regime — separately on dev and holdout. This diagnostic shows where the edge comes from and whether it is stable out of sample.

In [ ]:
# ---------------------------------------------------------------------------
# Sharpe by regime — computed inline, no helper function
# ---------------------------------------------------------------------------

def _sharpe_by_regime_table(strat_df, ret_col, label):
    """Print Sharpe by regime for a given return column."""
    rows = []
    for r in ["calm", "transition", "stress"]:
        sub = strat_df.loc[strat_df["regime"] == r, ret_col]
        if len(sub) < 5:
            continue
        rows.append({
            "regime": r,
            "n_days": len(sub),
            "ann_sharpe": round(np.sqrt(252) * sub.mean() / sub.std(), 2) if sub.std() > 0 else 0.0,
            "mean_ret_bps": round(sub.mean() * 10000, 1),
            "daily_vol_pct": round(sub.std() * 100, 2),
        })
    return pd.DataFrame(rows).set_index("regime")


print("=== Regime-Ridge ===")
print("\nDev:")
display(_sharpe_by_regime_table(dev_strat, "ret_ridge", "dev"))
print("\nHoldout:")
display(_sharpe_by_regime_table(hold_strat, "ret_ridge", "holdout"))

print("\n=== Regime-RF ===")
print("\nDev:")
display(_sharpe_by_regime_table(dev_strat, "ret_rf", "dev"))
print("\nHoldout:")
display(_sharpe_by_regime_table(hold_strat, "ret_rf", "holdout"))

print("\n=== Ridge uncond. ===")
print("\nDev:")
display(_sharpe_by_regime_table(dev_strat, "ret_ridge_unc", "dev"))
print("\nHoldout:")
display(_sharpe_by_regime_table(hold_strat, "ret_ridge_unc", "holdout"))

### Sharpe stability — interpretation

**Regime-Ridge:** On the dev, stress is the strongest regime (Sharpe ~1.4), followed by transition (~0.9) and calm (~0.8). On the holdout, stress holds up (Sharpe ~1.6), calm drops to ~0.5, and transition flips negative (~−0.7). The transition regime is unstable — the strategy's signal mix (vix_lag + mom_5d) does not generalise in this intermediate state. The overall holdout Sharpe (~0.6) is carried entirely by stress days.

**Regime-RF:** Dev Sharpe is implausibly uniform across regimes (~3–4 everywhere) — a hallmark of overfitting. On the holdout, stress still leads (~2.1), calm is reasonable (~1.7), and transition is the weakest (~0.8) but still positive. The RF's non-linearity helps in transition where Ridge fails, but the uniformly high dev numbers remain suspicious.

**Unconditional Ridge:** Interestingly, on the holdout the unconditional Ridge has its *highest* Sharpe in calm (~2.5) — the opposite of the regime-Ridge. This suggests the regime split may be *hurting* the calm-regime signal by fitting a separate model on a subset rather than pooling all data. In stress, the unconditional Ridge (Sharpe ~0.6) is weaker than the regime-Ridge (~1.6), confirming that regime conditioning helps specifically in stress.

**Bottom line:** The regime layer's value is concentrated in stress days. For calm and transition, the unconditional model is at least as good. A hybrid approach — unconditional model for calm/transition, regime-specific model for stress — might capture the best of both.

## 13. Statistical significance

In [ ]:
X_const = np.ones((len(hold_strat), 1))

print("Holdout significance (Newey-West, HAC with 10 lags):\n")
for label, ret_col in [("Buy-and-hold", "target"),
                        ("Ridge uncond.", "ret_ridge_unc"),
                        ("RF uncond.", "ret_rf_unc"),
                        ("Regime-Ridge", "ret_ridge"),
                        ("Regime-RF", "ret_rf")]:
    nw = sm.OLS(hold_strat[ret_col].values, X_const).fit(
        cov_type="HAC", cov_kwds={"maxlags": 10})
    print(f"  {label:20s}: mean={nw.params[0]*100:+.4f}%/day, "
          f"t={nw.tvalues[0]:+.2f}, p={nw.pvalues[0]:.4f}")

Only the regime-RF reaches conventional significance (t ≈ 2.4, p ≈ 0.017) — but given the overfitting concerns flagged earlier, this should be interpreted cautiously. The regime-Ridge (t ≈ 0.9, p ≈ 0.35) is not statistically distinguishable from zero.

With ~580 holdout observations, even a daily Sharpe of 1.0 translates to a t-statistic of only ~1.5. Significance at p < 0.05 requires a very strong or very long signal. The Newey-West adjustment (10 lags) accounts for serial correlation but does not help with small samples.

## Final remarks

**The regime detection is sound.** BIC selects k=3, the three regimes (calm, transition, stress) have distinct profiles that are stable across dev and holdout. Feature profiles (VIX ≈ 14/21/31, breadth ≈ 0.65/0.46/0.28) are nearly identical across samples. The GMM generalises.

**The RF overfits despite constrained hyperparameters.** In-sample correlations of 0.22–0.28 per regime, and a dev Sharpe of ~3.5, are implausible when the underlying signal correlations are ~5%. The RF memorises tree splits on small per-regime samples (772–1305 obs). The 55% Sharpe drop from dev to holdout confirms this. Constraining max_depth to 2–3 and min_samples_leaf to 50–120 is not sufficient to prevent overfitting in this low signal-to-noise environment.

**The regime layer does not add value after transaction costs.** Regime-conditioned models generate ~60–70 more annual turnover units than unconditional ones, translating to a Sharpe haircut ~0.2–0.3 larger. On the holdout, the unconditional RF (Sharpe net ~0.9) and unconditional Ridge (Sharpe net ~0.35) both outperform their regime-conditioned counterparts (regime-RF net ~1.0, regime-Ridge net ~0.03). The marginal signal improvement from regime conditioning is consumed by the extra cost of regime-switching.

**The regime layer's value is concentrated in stress.** The by-regime Sharpe analysis shows that regime conditioning specifically helps in stress (regime-Ridge Sharpe 1.6 vs unconditional Ridge 0.6 in stress). In calm and transition, the unconditional model matches or beats the regime-conditioned one. A practical implementation could use the unconditional model as default and switch to the stress-specific model only when the GMM classifies the current state as stress.

**Ridge vs RF: Ridge is preferred.** The Ridge is more regularised (alpha 5000–10000), more interpretable (explicit coefficients), less prone to overfitting, and cheaper to trade (smoother predictions → lower turnover). The RF's non-linearity does not compensate for its overfitting risk in this low-SNR environment.

**What I would add next:**

- Walk-forward retraining (monthly/quarterly) to adapt to signal decay and reduce the in-sample / out-of-sample gap
- Hybrid strategy: unconditional model for calm/transition, regime-specific model for stress only
- Vol-targeting benchmark (exposure = target_vol / realised_vol) as a simpler risk-management alternative that avoids signal prediction entirely